In [ ]:
import os
import numpy as np
import pandas as pd

DATA_DIR = "../..data/cleansed"

FLIGHTS_PATH = os.path.join(DATA_DIR, "all_flights_2018-2022.parquet")
WEATHER_PATH = os.path.join(DATA_DIR, "weather_airports_2018_2022_CLEAN.parquet")  

print("Flights exists?", os.path.exists(FLIGHTS_PATH), FLIGHTS_PATH)
print("Weather exists?", os.path.exists(WEATHER_PATH), WEATHER_PATH)

In [ ]:
flight_cols = [
    "FlightDate",
    "CRSDepTime",
    "Distance",
    "Origin",
    "Cancelled",
    "DepDelayMinutes",
]

flights = pd.read_parquet(FLIGHTS_PATH, columns=flight_cols)
print("flights:", flights.shape)
flights.head()

In [ ]:
flights["FlightDate"] = pd.to_datetime(flights["FlightDate"], errors="coerce")
flights = flights.dropna(subset=["FlightDate"]).copy()

# Dropping 2020 (COVID anomaly)
flights = flights[flights["FlightDate"].dt.year != 2020].copy()

# Types
flights["Cancelled"] = flights["Cancelled"].astype(bool)
flights["DepDelayMinutes"] = pd.to_numeric(flights["DepDelayMinutes"], errors="coerce").fillna(0)

flights["target"] = np.select(
    [
        flights["Cancelled"] == True,
        (flights["Cancelled"] == False) & (flights["DepDelayMinutes"] >= 15),
    ],
    ["Cancelled", "Delayed"],
    default="On time",
)

print("After cleaning:", flights.shape)
print(flights["target"].value_counts(normalize=True))

In [ ]:
flights["CRSDepTime"] = pd.to_numeric(flights["CRSDepTime"], errors="coerce")
flights["dep_hour"] = (flights["CRSDepTime"] // 100).astype("Int64")

flights["month"] = flights["FlightDate"].dt.month.astype("Int64")
flights["day_of_week"] = flights["FlightDate"].dt.dayofweek.astype("Int64")

flights = flights.dropna(subset=["dep_hour"]).copy()
flights = flights[(flights["dep_hour"] >= 0) & (flights["dep_hour"] <= 23)].copy()

print("After FE:", flights.shape)

In [ ]:
USE_WEATHER = os.path.exists(WEATHER_PATH)
print("USE_WEATHER =", USE_WEATHER)

if USE_WEATHER:
    weather_cols = ["airport_code","valid","tmpf","vsby","sknt","p01i","relh","gust"]
    weather = pd.read_parquet(WEATHER_PATH, columns=weather_cols)
    print("weather:", weather.shape)

    weather["valid"] = pd.to_datetime(weather["valid"], errors="coerce")
    weather = weather.dropna(subset=["valid"]).copy()
    weather["date"] = weather["valid"].dt.date
    weather["hour"] = weather["valid"].dt.hour

    weather_hourly = (
        weather.groupby(["airport_code", "date", "hour"], as_index=False)
        .agg({
            "tmpf": "mean",
            "vsby": "mean",
            "sknt": "mean",
            "p01i": "sum",
            "relh": "mean",
            "gust": "max",
        })
    )

    flights["date"] = flights["FlightDate"].dt.date
    flights = flights.merge(
        weather_hourly,
        left_on=["Origin", "date", "dep_hour"],
        right_on=["airport_code", "date", "hour"],
        how="left",
    ).drop(columns=["airport_code", "hour"], errors="ignore")

    print("Weather coverage tmpf (% non-null):", (1 - flights["tmpf"].isna().mean()) * 100)
else:
    print("Skipping weather merge because weather parquet is not in data/cleansed yet.")

In [ ]:
BASE_NUM = ["Distance", "dep_hour", "day_of_week", "month"]
WEATHER_NUM = ["tmpf", "vsby", "sknt", "p01i", "relh", "gust"]
CAT_COLS = ["Origin"]

NUM_COLS = BASE_NUM + (WEATHER_NUM if ("tmpf" in flights.columns) else [])

# Keeping FlightDate for splitting only (not a feature)
keep_cols = ["FlightDate", "target"] + NUM_COLS + CAT_COLS
df = flights[keep_cols].copy()

print("df:", df.shape)
print(df["target"].value_counts(normalize=True))

In [ ]:
df = df.sort_values("FlightDate")
cutoff = pd.Timestamp("2022-01-01")

# use a manageable slice
train_part = df[df["FlightDate"] < cutoff].tail(300_000)
test_part  = df[df["FlightDate"] >= cutoff].head(100_000)

df_small = pd.concat([train_part, test_part], axis=0)

print("Using subset:", df_small.shape)
print("Train rows:", train_part.shape[0], "Test rows:", test_part.shape[0])
print(df_small["target"].value_counts(normalize=True))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

train_df = df_small[df_small["FlightDate"] < cutoff].copy()
test_df  = df_small[df_small["FlightDate"] >= cutoff].copy()

X_train = train_df[NUM_COLS + CAT_COLS]
y_train = train_df["target"]

X_test = test_df[NUM_COLS + CAT_COLS]
y_test = test_df["target"]

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])

prep = ColumnTransformer([
    ("num", num_pipe, NUM_COLS),
    ("cat", cat_pipe, CAT_COLS),
])

lr = LogisticRegression(
    multi_class="multinomial",
    solver="saga",
    max_iter=300,
    n_jobs=-1,
    class_weight="balanced"
)

model = Pipeline([("prep", prep), ("lr", lr)])

print("✅ Model ready. Train:", X_train.shape, "Test:", X_test.shape)

In [ ]:
model.fit(X_train, y_train)
print("✅ Done fitting.")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

pred = model.predict(X_test)

print(classification_report(y_test, pred, digits=3))
print("Confusion matrix (labels = On time, Delayed, Cancelled):")
print(confusion_matrix(y_test, pred, labels=["On time", "Delayed", "Cancelled"]))

In [ ]:
feature_names = model.named_steps["prep"].get_feature_names_out()
classes = model.named_steps["lr"].classes_
coefs = model.named_steps["lr"].coef_  # (n_classes, n_features)

coef_df = pd.DataFrame(coefs.T, index=feature_names, columns=classes)

for c in classes:
    print("\n====", c, "====")
    display(coef_df[c].sort_values(ascending=False).head(10))
    display(coef_df[c].sort_values(ascending=True).head(10))